# Sandpile on Free Associations Network

Experiment: compare **Random** vs **Targeted** grain addition on a Free Association network using the Abelian Sandpile with stochastic dissipation (f=10⁻⁴).

**Measures:**
1. Avalanche Frequency — fraction of iterations producing at least one toppling
2. Avalanche Size — total grains moved per avalanche (Σ deg of toppled nodes)
3. Unique Toppled Nodes — number of distinct nodes involved per avalanche
4. Total Topplings — toppling events per avalanche (with repetitions)
5. Target Involvement — how often the target node is involved in avalanches

In [ ]:
import sys
sys.path.insert(0, '..')

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm

from SpreadPy.Models.sandpile import SandpileSpreading

np.random.seed(42)

## 1. Load Free Associations Network

In [ ]:
# Build graph from edge list
g = nx.Graph()
with open('../toy_data/mental_lexicon_2/FreeAssociations.txt') as f:
    for line in f:
        parts = line.strip().split('	')
        if len(parts) == 2:
            g.add_edge(parts[0], parts[1])

# Remove self-loops
g.remove_edges_from(nx.selfloop_edges(g))

# Remove degree-0 nodes
isolates = list(nx.isolates(g))
g.remove_nodes_from(isolates)
print(f"Removed {len(isolates)} isolated nodes")

# Take largest connected component
largest_cc = max(nx.connected_components(g), key=len)
g = g.subgraph(largest_cc).copy()

print(f"Nodes: {g.number_of_nodes()}")
print(f"Edges: {g.number_of_edges()}")
print(f"Avg degree: {2 * g.number_of_edges() / g.number_of_nodes():.1f}")

## 2. Load Dataset & Sample 100 Pairs

In [ ]:
df = pd.read_excel('../data/all naming subjects.xlsx')

# Keep only related pairs (primecond=1), lowercase, deduplicate
related = df[df['primecond'] == 1.0][['prime', 'target']].drop_duplicates()
related['prime'] = related['prime'].str.lower()
related['target'] = related['target'].str.lower()

# Keep only pairs where both words exist in the network
nodes = set(g.nodes())
related = related[(related['prime'].isin(nodes)) & (related['target'].isin(nodes))]
print(f"Related pairs with both words in network: {len(related)}")

# Sample N pairs
pairs = related.sample(n=100, random_state=42).reset_index(drop=True)
pairs_cue = pairs.iloc[:75]
pairs_random = pairs.iloc[75:]
print(f"Sampled {len(pairs)} pairs (75 CUE, 25 RANDOM)")
pairs.head(10)

## 3. Run Experiments

For each pair, run two sandpile simulations with N iterations:
- **Cue** (75 pairs): always add grain to the cue (prime) node
- **Random** (25 pairs): add grain to a random node each step
In both conditions we track if the target node is involved.

In [ ]:
N_ITERATIONS = 10000  # iterations per simulation
F = 1e-4             # dissipation probability

def run_sandpile(graph, n_iter, target_node=None, cue_node=None):
    """
    Run a sandpile simulation and collect avalanche metrics.
    
    :param graph: networkx graph
    :param n_iter: number of iterations
    :param target_node: if set, track whether this node is involved in avalanches
    :param cue_node: if set, always add grain to this node (targeted mode)
    :return: dict with collected metrics
    """
    model = SandpileSpreading(graph, f=F)
    
    avalanche_sizes = []
    avalanche_topplings = []
    unique_toppled = []
    target_involved = []  # True/False per iteration
    has_avalanche = []
    
    # iteration 0 (initial config)
    model.iteration()
    
    for _ in range(n_iter):
        result = model.iteration(node=cue_node)
        
        avalanche_sizes.append(result['avalanche_size'])
        avalanche_topplings.append(result['total_topplings'])
        unique_toppled.append(result['unique_toppled_nodes'])
        has_avalanche.append(result['has_avalanche'])
        
        if target_node is not None:
            target_involved.append(target_node in result['toppled_nodes'])
    
    return {
        'avalanche_sizes': avalanche_sizes,
        'avalanche_topplings': avalanche_topplings,
        'unique_toppled': unique_toppled,
        'target_involved': target_involved,
        'has_avalanche': has_avalanche,
    }

In [ ]:
results_random = []
results_cue = []

for idx, row in tqdm(pairs_cue.iterrows(), total=len(pairs_cue), desc="Running CUE"):
    cue = row['prime']
    target = row['target']
    
    res_t = run_sandpile(g, N_ITERATIONS, target_node=target, cue_node=cue)
    results_cue.append(res_t)

for idx, row in tqdm(pairs_random.iterrows(), total=len(pairs_random), desc="Running RANDOM"):
    cue = row['prime']
    target = row['target']
    
    res_r = run_sandpile(g, N_ITERATIONS, target_node=target, cue_node=None)
    results_random.append(res_r)
    

### Checkpoint: Salvataggio e Caricamento dei dati
Queste celle permettono di salvare i risultati delle simulazioni su disco e ricaricarli in un secondo momento, utile per modificare l'analisi senza dover rifare i lunghi calcoli.

In [ ]:
import pickle
import os

# Assicurati che la cartella esista
os.makedirs('../data', exist_ok=True)

# Salva i risultati
with open('../data/sandpile_results.pkl', 'wb') as f:
    pickle.dump({'random': results_random, 'cue': results_cue, 'pairs_random': pairs_random, 'pairs_cue': pairs_cue}, f)
print("Risultati salvati in ../data/sandpile_results.pkl")

In [ ]:
import pickle

# Decommenta le righe seguenti per ricaricare i dati senza rifare le simulazioni in alto
with open('../data/sandpile_results.pkl', 'rb') as f:
    data = pickle.load(f)
    results_random = data['random']
    results_cue = data['cue']
    pairs_random = data.get('pairs_random', None)
    pairs_cue = data.get('pairs_cue', None)
print(f"Risultati caricati con successo: {len(results_random)} simulazioni random, {len(results_cue)} cue.")

## 4. Analysis



In [ ]:
import powerlaw
import numpy as np
import matplotlib.pyplot as plt

# Collect non-zero avalanche sizes
sizes_random = [s for r in results_random for s in r['avalanche_sizes'] if s > 0]
sizes_cue = [s for r in results_cue for s in r['avalanche_sizes'] if s > 0]

# Collect non-zero topplings
topplings_random = [t for r in results_random for t in r['avalanche_topplings'] if t > 0]
topplings_cue = [t for r in results_cue for t in r['avalanche_topplings'] if t > 0]

# Collect non-zero unique toppled
unique_random = [u for r in results_random for u in r['unique_toppled'] if u > 0]

# Totals (Random + Cue combined)
sizes_total = sizes_random + sizes_cue
topplings_total = topplings_random + topplings_cue


#### Avalanche Size (Total) — CCDF + Power-Law Fit

Distribuzione complessiva delle avalanche sizes (Random + Cue combinati).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

fit_size_total = powerlaw.Fit(sizes_total, verbose=False)
fit_size_total.plot_ccdf(ax=ax, color='forestgreen', linewidth=2, label='All data (empirical)')
fit_size_total.power_law.plot_ccdf(ax=ax, color='forestgreen', linestyle='--', linewidth=2,
                                    label=f'Power-law fit (α={fit_size_total.power_law.alpha:.2f}, x_min={fit_size_total.power_law.xmin:.0f})')

ax.set_xlabel('Avalanche Size (grains moved)', fontsize=12)
ax.set_ylabel('CCDF  P(X ≥ x)', fontsize=12)
ax.set_title('Avalanche Size (Total) — CCDF with Power-Law Fit', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

#### Total Topplings (Total) — CCDF + Power-Law Fit

Distribuzione complessiva dei total topplings (Random + Cue combinati).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

fit_top_total = powerlaw.Fit(topplings_total, verbose=False)
fit_top_total.plot_ccdf(ax=ax, color='darkviolet', linewidth=2, label='All data (empirical)')
fit_top_total.power_law.plot_ccdf(ax=ax, color='darkviolet', linestyle='--', linewidth=2,
                                   label=f'Power-law fit (α={fit_top_total.power_law.alpha:.2f}, x_min={fit_top_total.power_law.xmin:.0f})')

ax.set_xlabel('Total Topplings', fontsize=12)
ax.set_ylabel('CCDF  P(X ≥ x)', fontsize=12)
ax.set_title('Total Topplings (Total) — CCDF with Power-Law Fit', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

#### Avalanche Size — CCDF + Power-Law Fit

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Random
fit_size_random = powerlaw.Fit(sizes_random, verbose=False)
fit_size_random.plot_ccdf(ax=ax, color='steelblue', linewidth=2, label='Random (empirical)')
fit_size_random.power_law.plot_ccdf(ax=ax, color='steelblue', linestyle='--', linewidth=2,
                                     label=f'Random fit (α={fit_size_random.power_law.alpha:.2f}, x_min={fit_size_random.power_law.xmin:.0f})')

# Targeted
fit_size_targeted = powerlaw.Fit(sizes_targeted, verbose=False)
fit_size_targeted.plot_ccdf(ax=ax, color='coral', linewidth=2, label='Targeted (empirical)')
fit_size_targeted.power_law.plot_ccdf(ax=ax, color='coral', linestyle='--', linewidth=2,
                                       label=f'Targeted fit (α={fit_size_targeted.power_law.alpha:.2f}, x_min={fit_size_targeted.power_law.xmin:.0f})')

ax.set_xlabel('Avalanche Size (grains moved)', fontsize=12)
ax.set_ylabel('CCDF  P(X ≥ x)', fontsize=12)
ax.set_title('Avalanche Size — CCDF with Power-Law Fit', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Random
fit_top_random = powerlaw.Fit(topplings_random, verbose=False)
fit_top_random.plot_ccdf(ax=ax, color='steelblue', linewidth=2, label='Random (empirical)')
fit_top_random.power_law.plot_ccdf(ax=ax, color='steelblue', linestyle='--', linewidth=2,
                                    label=f'Random fit (α={fit_top_random.power_law.alpha:.2f}, x_min={fit_top_random.power_law.xmin:.0f})')

# Targeted
fit_top_targeted = powerlaw.Fit(topplings_targeted, verbose=False)
fit_top_targeted.plot_ccdf(ax=ax, color='coral', linewidth=2, label='Targeted (empirical)')
fit_top_targeted.power_law.plot_ccdf(ax=ax, color='coral', linestyle='--', linewidth=2,
                                      label=f'Targeted fit (α={fit_top_targeted.power_law.alpha:.2f}, x_min={fit_top_targeted.power_law.xmin:.0f})')

ax.set_xlabel('Total Topplings', fontsize=12)
ax.set_ylabel('CCDF  P(X ≥ x)', fontsize=12)
ax.set_title('Total Topplings — CCDF with Power-Law Fit', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()


#### Total Topplings — CCDF + Power-Law Fit

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Random
fit_uniq_random = powerlaw.Fit(unique_random, verbose=False)
fit_uniq_random.plot_ccdf(ax=ax, color='steelblue', linewidth=2, label='Random (empirical)')
fit_uniq_random.power_law.plot_ccdf(ax=ax, color='steelblue', linestyle='--', linewidth=2,
                                     label=f'Random fit (α={fit_uniq_random.power_law.alpha:.2f}, x_min={fit_uniq_random.power_law.xmin:.0f})')

# Targeted
fit_uniq_targeted = powerlaw.Fit(unique_targeted, verbose=False)
fit_uniq_targeted.plot_ccdf(ax=ax, color='coral', linewidth=2, label='Targeted (empirical)')
fit_uniq_targeted.power_law.plot_ccdf(ax=ax, color='coral', linestyle='--', linewidth=2,
                                       label=f'Targeted fit (α={fit_uniq_targeted.power_law.alpha:.2f}, x_min={fit_uniq_targeted.power_law.xmin:.0f})')

ax.set_xlabel('Unique Toppled Nodes', fontsize=12)
ax.set_ylabel('CCDF  P(X ≥ x)', fontsize=12)
ax.set_title('Unique Toppled Nodes — CCDF with Power-Law Fit', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()


In [ ]:
prob_avalanche_cue = np.mean([r['has_avalanche'] for r in results_cue], axis=0)
prob_avalanche_random = np.mean([r['has_avalanche'] for r in results_random], axis=0)

window = 100
prob_smooth_cue = pd.Series(prob_avalanche_cue).rolling(window=window, min_periods=1).mean()
prob_smooth_random = pd.Series(prob_avalanche_random).rolling(window=window, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prob_smooth_cue, color='darkorange', linewidth=2, label='CUE')
ax.plot(prob_smooth_random, color='steelblue', linewidth=2, label='RANDOM')
ax.set_xlabel('Timestamp (iterazione)', fontsize=12)
ax.set_ylabel('P(avalanche)', fontsize=12)
ax.set_title(f'Transizione di fase — Probabilità di valanga nel tempo (media mobile w={window})', fontsize=14)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()